# LangChain: Models, Messages & Structured Output

## Outline
* نصب و setup
* مقایسه LangChain با litellm / openai یا ollama مستقیم
* init_chat_model — اتصال به هر provider
* Messages — انواع پیام‌ها
* Prompt Templates
* Streaming
* Structured Output با Pydantic
* Token Usage


## ۱. نصب

In [ ]:
# نصب پکیج‌های مورد نیاز
# pip install -U langchain langchain-openai langchain-ollama python-dotenv


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

## ۲. چرا LangChain؟

- **LangChain**: علاوه بر یکپارچگی provider، ابزارهای کامل برای ساخت agent، memory، RAG، و evaluation


In [4]:
# روش قدیمی — openai مستقیم
# from openai import OpenAI
# client = OpenAI()
# response = client.chat.completions.create(
#     model="gpt-4o",
#     messages=[{"role": "user", "content": "Hello"}]
# )
# print(response.choices[0].message.content)

# روش LangChain — یکسان برای همه providerها
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5.2", model_provider="openai", temperature=0)
response = model.invoke("Hello! What can you do?")
print(response.content)


I can help with a wide range of tasks. Here are the main things I’m useful for:

- **Answer questions & explain concepts**: math, science, history, programming, finance, etc.
- **Write & edit**: emails, resumes, cover letters, reports, blog posts, scripts; proofreading and rewrites for tone/clarity.
- **Brainstorm & plan**: ideas for projects, lessons, content calendars, trip itineraries, study plans.
- **Coding help**: debug errors, design algorithms, write snippets, review code, explain stack traces (Python/JS/SQL/etc.).
- **Data & analysis**: summarize documents, compare options, create tables, basic stats, interpret results.
- **Work with images** (if you share one): describe what’s in it, extract text, troubleshoot diagrams, help identify objects.
- **Decision support**: weigh tradeoffs, make checklists, craft questions to ask, evaluate pros/cons.

If you tell me what you’re working on (and your goal/audience), I’ll jump in.


## ۳. init_chat_model — اتصال به هر Provider

مهم‌ترین تغییر نسبت به LangChain قدیمی: دیگه نیازی به import جداگانه برای هر provider نیست.


In [6]:
from langchain.chat_models import init_chat_model

# Ollama
model_ollama = init_chat_model("gemma3:4b", model_provider="ollama", temperature=0)

# Anthropic (نیاز به: pip install langchain-anthropic)
# model_anthropic = init_chat_model("claude-sonnet-4-6", model_provider="anthropic")

# Ollama — مدل local (نیاز به: pip install langchain-ollama)
# model_ollama = init_chat_model("llama3.2", model_provider="ollama")

# OpenAI-compatible (مثل litellm proxy یا vLLM)
# model_custom = init_chat_model(
#     model="your-model",
#     model_provider="openai",
#     base_url="http://localhost:11434/v1",
#     api_key="dummy"
# )

print(type(model_ollama))


<class 'langchain_ollama.chat_models.ChatOllama'>


In [7]:
a = model_ollama.invoke("سلام حالت چه طوره")
a.content

'سلام! من خوبم، ممنون که پرسیدی. شما چطورید؟ امیدوارم شما هم حالتون خوب باشه. 😊\n'

In [8]:
# پارامترهای مهم
model = init_chat_model(
    "gemma4:e4b",
    model_provider="ollama",
    temperature=0.7,       # خلاقیت (0 تا 2)
    max_tokens=500,        # حداکثر توکن خروجی
    timeout=30,            # timeout به ثانیه
    max_retries=3,         # تعداد retry در صورت خطا
)

## ۴. Messages — انواع پیام‌ها

در LangChain همه چیز با پیام کار می‌کنه. این مستقیماً با OpenAI API مپ می‌شه.


In [10]:
model = init_chat_model("gpt-5.2", model_provider="openai", temperature=0)

In [11]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# سه نوع اصلی پیام
system = SystemMessage("You are a helpful Persian-speaking assistant.")
human = HumanMessage("سلام! اسم من علی است.")
ai = AIMessage("سلام علی! چطور می‌تونم کمکت کنم؟")

# ارسال conversation کامل
conversation = [system, human, ai, HumanMessage("اسم من چیه؟")]
response = model.invoke(conversation)
print(response.content)
print(f"\nنوع response: {type(response)}")

اسم شما **علی** است.

نوع response: <class 'langchain_core.messages.ai.AIMessage'>


In [13]:
# می‌تونید از dict هم استفاده کنید (مثل OpenAI API مستقیم)
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
]
response = model.invoke(messages)
print(response.content)


2 + 2 = 4


In [15]:
# metadata و token usage در response
response = model.invoke("Tell me a joke")
print(f"Content: {response.content}")
print(f"Model: {response.response_metadata.get('model_name', 'N/A')}")
print(f"Usage: {response.usage_metadata}")


Content: Why don’t skeletons fight each other?  
They don’t have the guts.
Model: gpt-5.2-2025-12-11
Usage: {'input_tokens': 10, 'output_tokens': 20, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [22]:
response

AIMessage(content='Why don’t skeletons fight each other?  \nThey don’t have the guts.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 10, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.2-2025-12-11', 'system_fingerprint': None, 'id': 'chatcmpl-ELAaKFlNrzxQ0ZFi6HEXy3h6M2Ocp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a077ad-fa07-7622-bcf5-24785c78bacf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 20, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## ۵. Prompt Templates

Prompt template مثل یه تابع است که ورودی می‌گیره و prompt آماده برمی‌گردونه.


In [24]:
from langchain_core.prompts import ChatPromptTemplate

# ساده‌ترین حالت
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that translates {input_language} to {output_language}.
     You must only translate the text with no description, no explanation, no additional text."""),
    ("human", "{text}")
])

# پر کردن template
messages = prompt.invoke({"input_language":"English", "output_language":"Persian", "text":"Hi, How are you Ali!"})

print(messages)


messages=[SystemMessage(content='You are a helpful assistant that translates English to Persian.\n     You must only translate the text with no description, no explanation, no additional text.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi, How are you Ali!', additional_kwargs={}, response_metadata={})]


In [26]:
response = model.invoke(messages)
print(response.content)

سلام، حالت چطوره علی!


In [28]:
messages = prompt.invoke({
    "input_language": "Persian",
    "output_language": "Arabic",
    "text": "اسم شما چیه و چند سالتونه؟"
})
response = model.invoke(messages)
print(response.content)

ما اسمك وكم عمرك؟


In [30]:
# ساده‌ترین حالت
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that translates English to Persian.
     You must only translate the text with no description, no explanation, no additional text."""),
    ("human", "{text}")
])

# پر کردن template
messages = prompt.invoke("Hi, How are you Ali!")

print(messages)


messages=[SystemMessage(content='You are a helpful assistant that translates English to Persian.\n     You must only translate the text with no description, no explanation, no additional text.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi, How are you Ali!', additional_kwargs={}, response_metadata={})]


In [32]:
simple_chain = prompt | model
r = simple_chain.invoke("Hello, how are you")
r.content

'سلام، حال شما چطور است؟'

## 5 . LCEL: LangChain Expression Language

**Building a `chain` with the `|` operator in LCEL:** Connects the prompt and model, then invokes the chain with a dictionary of prompt variables and prints the model's final output.

In [34]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that translates {input_language} to {output_language}.
     You must only translate the text with no description, no explanation, no additional text."""),
    ("human", "{text}")
])

In [36]:
# استفاده در یک chain با LCEL (pipe operator)
chain = prompt | model 

response = chain.invoke({
    "input_language": "English",
    "output_language": "Persian",
    "text": "LangChain is a powerful framework for building LLM applications."
})
print(response.content)


لانگ‌چین یک چارچوب قدرتمند برای ساخت برنامه‌های کاربردی مبتنی بر مدل‌های زبانی بزرگ (LLM) است.


In [37]:
response = chain.invoke({
    "input_language": "English",
    "output_language": "Arabic",
    "text": "LangChain is a powerful framework for building LLM applications."
})
print(response.content)

لانغتشين هو إطار عمل قوي لبناء تطبيقات النماذج اللغوية الكبيرة.


In [40]:
# Few-shot prompt — دادن مثال به مدل
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment analyzer. Reply with only: POSITIVE, NEGATIVE, or NEUTRAL"),
    ("human", "I love this product!"),
    ("ai", "POSITIVE"),
    ("human", "This is terrible."),
    ("ai", "NEGATIVE"),
    ("human", "It's okay, nothing special."),
    ("ai", "NEUTRAL"),
    ("human", "{text}"),
])

chain = few_shot_prompt | model
result = chain.invoke("خدایی قیمتش خیلی خوبه اما جنس بدنش خیلی ضعیفه")
print(result.content)


NEUTRAL


In [41]:
result = chain.invoke("Best purchase I've made this year!")
print(result.content)

POSITIVE


## ۶. Streaming

با streaming می‌تونید توکن‌ها رو همزمان با تولید نشون بدید — مثل ChatGPT.


In [45]:
# streaming ساده
print("خروجی streaming:")
for chunk in model.stream("یک بلاگ پست 2 پاراگرافی در مورد rag بنویس (retrival augumented generation)"):
    print(chunk.content, end="", flush=True)
print()  # newline


خروجی streaming:
RAG یا **Retrieval-Augmented Generation** (تولید تقویت‌شده با بازیابی) رویکردی در هوش مصنوعی است که تلاش می‌کند ضعف اصلی مدل‌های زبانی را پوشش دهد: این‌که حافظه‌ی ثابت دارند، ممکن است اطلاعاتشان قدیمی باشد و گاهی هم «با اطمینان» چیزی را بسازند که واقعیت ندارد. در RAG، قبل از این‌که مدل پاسخ نهایی را تولید کند، ابتدا یک مرحله‌ی **بازیابی (Retrieval)** انجام می‌شود؛ یعنی سیستم از میان منابع بیرونی مثل پایگاه‌دانش شرکت، اسناد PDF، ویکی داخلی، دیتابیس‌ها یا حتی وب، چند متن مرتبط را پیدا می‌کند و به‌عنوان «زمینه» به مدل می‌دهد. سپس مرحله‌ی **تولید (Generation)** با تکیه بر همین شواهد انجام می‌شود. نتیجه معمولاً پاسخ‌هایی دقیق‌تر، قابل استنادتر و نزدیک‌تر به محتوای واقعی سازمان است، چون مدل به جای تکیه‌ی صرف بر دانش عمومیِ یادگرفته‌شده، به اطلاعات بازیابی‌شده هم اتکا می‌کند.

کاربردهای RAG بسیار گسترده‌اند: از ساخت چت‌بات‌های پشتیبانی مشتری که به مستندات واقعی محصول متکی‌اند، تا دستیارهای تحلیل قرارداد، جست‌وجوی معنایی در دانش سازمانی، یا پاسخ‌گویی به پرسش‌های تخصصی روی مجمو

In [49]:
# batch — ارسال چند request موازی
responses = model.batch([
    "What is Python?",
    "What is JavaScript?",
    "What is Rust?",
])
for i, r in enumerate(responses):
    print(f"Response {i+1}: {r.content[:80]}...")

Response 1: Python is a high-level, general-purpose programming language known for its clear...
Response 2: JavaScript is a high-level programming language used to make web pages interacti...
Response 3: Rust is a modern programming language designed for building fast, reliable softw...


## 7. Structured Output with Pydantic

Instead of parsing text, force the model to directly output structured data.
This replaces the legacy `ResponseSchema` and `StructuredOutputParser`.

In [69]:
from pydantic import BaseModel, Field
from typing import List

# تعریف schema با Pydantic
class ProductReview(BaseModel):
    """اطلاعات استخراج‌شده از یک نظر محصول"""
    gift: bool = Field(description="آیا محصول به عنوان هدیه خریداری شده؟")
    delivery_days: int = Field(description="چند روز تحویل داد؟ اگر نبود -1")
    price_value: List[str] = Field(description="جملاتی درباره قیمت یا ارزش محصول")

# ساخت model با structured output
structured_model = model.with_structured_output(ProductReview)

In [71]:
review = """
من اینو به عنوان روز مادر خریدم؛ اما 3 هفته طول کشید دستم برسه. قیمتش خوبه اما ارسال کالا افتضاحه """

result = structured_model.invoke(f"Extract info from this review:{review}")

In [72]:
result

ProductReview(gift=True, delivery_days=21, price_value=['قیمتش خوبه'])

In [75]:

print(f"Gift: {result.gift}")
print(f"Delivery days: {result.delivery_days}")
print(f"Price comments: {result.price_value}")
print(f"\nType: {type(result)}")


Gift: True
Delivery days: 21
Price comments: ['قیمتش خوبه']

Type: <class '__main__.ProductReview'>


In [78]:
lite_model = init_chat_model("gemma4:e4b", model_provider="ollama", temperature=0)
# ساخت model با structured output
lite_model_structured_model = lite_model.with_structured_output(ProductReview)
lite_model_structured_model.invoke(f"Extract info from this review: {review}, convert week or month or year into a day")

ProductReview(gift=True, delivery_days=21, price_value=['Good price', 'قیمتش خوبه'])

In [80]:
# مثال دیگر: استخراج اطلاعات ساختاریافته
class MovieInfo(BaseModel):
    """اطلاعات فیلم"""
    title: str = Field(description="عنوان فیلم")
    year: int = Field(description="سال ساخت")
    director: str = Field(description="کارگردان")
    rating: float = Field(description="امتیاز از 10")

movie_model = model.with_structured_output(MovieInfo)
result = movie_model.invoke("در مورد فیلم inception بهم بگو")
    
print(result)
print(f"\nTitle: {result.title}, Year: {result.year}")


title='Inception' year=2010 director='Christopher Nolan' rating=8.8

Title: Inception, Year: 2010


## ۸. Token Usage — ردیابی مصرف

برای کنترل هزینه مهمه.


In [82]:
from langchain_core.callbacks import get_usage_metadata_callback

model_gpt = init_chat_model("gpt-4o", model_provider="openai")

with get_usage_metadata_callback() as cb:
    model_gpt.invoke("Hello!")
    model_gpt.invoke("what is capital of Iran!")
    model_gpt.invoke("What is the capital of France?")

print(cb.usage_metadata)

{'gpt-4o-2024-08-06': {'input_tokens': 36, 'output_token_details': {'reasoning': 0, 'audio': 0}, 'total_tokens': 59, 'input_token_details': {'cache_read': 0, 'audio': 0}, 'output_tokens': 23}}


In [84]:
# usage در هر response هم موجوده
response = model.invoke("Explain machine learning in one sentence")
print(f"Input tokens:  {response.usage_metadata['input_tokens']}")
print(f"Output tokens: {response.usage_metadata['output_tokens']}")
print(f"Total tokens:  {response.usage_metadata['total_tokens']}")


Input tokens:  12
Output tokens: 29
Total tokens:  41


In [86]:
from langchain_core.callbacks import get_usage_metadata_callback

model_gpt4o = init_chat_model("gpt-4o", model_provider="openai")
model_gpt_5_2 = init_chat_model("gpt-5.2", model_provider="openai")

with get_usage_metadata_callback() as cb:
    model_gpt4o.invoke("Hello!")
    model_gpt4o.invoke("what is capital of Iran!")
    model_gpt_5_2.invoke("What is the capital of France?")

print(cb.usage_metadata)

{'gpt-4o-2024-08-06': {'input_tokens': 22, 'output_token_details': {'reasoning': 0, 'audio': 0}, 'input_token_details': {'cache_read': 0, 'audio': 0}, 'total_tokens': 38, 'output_tokens': 16}, 'gpt-5.2-2025-12-11': {'input_tokens': 13, 'output_tokens': 5, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}
